##  **BRANDS**

In [0]:
from pyspark.sql.types import StructType,StructField, StringType, IntegerType,FloatType, DateType, TimestampType
from pyspark.sql import functions as F

In [0]:
catalog_name = "ecommerce"
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show()

+----------+--------------+-------------+--------------------+--------------------+
|brand_code|    brand_name|category_code|        _source_file|         ingested_at|
+----------+--------------+-------------+--------------------+--------------------+
|      ACME|      AcmeTech|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      NOVW|     NovaWave |           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      ZNTH|        Zenith|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      BYTM|       ByteMax|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      ECOT|       EcoTone|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      SKYL|       SkyLink|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|     VOLT@|      VoltEdge|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      PHTX|      Photonix|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      URTL|    UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-11-16 14:

In [0]:
#remove extra trailing spaces from brand_code
df_silver = df_bronze.withColumn('brand_name', F.trim(F.col("brand_name"))) 
df_silver.show()

+----------+-----------+-------------+--------------------+--------------------+
|brand_code| brand_name|category_code|        _source_file|         ingested_at|
+----------+-----------+-------------+--------------------+--------------------+
|      ACME|   AcmeTech|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      NOVW|   NovaWave|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      ZNTH|     Zenith|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      BYTM|    ByteMax|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      ECOT|    EcoTone|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      SKYL|    SkyLink|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|     VOLT@|   VoltEdge|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      PHTX|   Photonix|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      URTL| UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      COTC| CottonClub|    

In [0]:
#now lets just say the brand_code should be alpha numeric meaning it should only have alphabets and numbers
#so we can use the regexp_replace function to replace any non alphanumeric characters with an empty string
from pyspark.sql.functions import regexp_replace
df_silver = df_silver.withColumn("brand_code",F.regexp_replace("brand_code","[^a-zA-Z0-9]",""))
df_silver.show()

+----------+-----------+-------------+--------------------+--------------------+
|brand_code| brand_name|category_code|        _source_file|         ingested_at|
+----------+-----------+-------------+--------------------+--------------------+
|      ACME|   AcmeTech|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      NOVW|   NovaWave|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      ZNTH|     Zenith|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      BYTM|    ByteMax|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      ECOT|    EcoTone|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      SKYL|    SkyLink|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      VOLT|   VoltEdge|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      PHTX|   Photonix|           CE|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      URTL| UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-11-16 14:04:...|
|      COTC| CottonClub|    

In [0]:
df_silver.select("category_code").distinct().show()

+-------------+
|category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|        BOOKS|
|          BKS|
|      GROCERY|
|         GRCY|
|          TOY|
|         TOYS|
|          SPT|
+-------------+



In [0]:
df_silver = df_silver.replace("BKS","BOOKS", subset='category_code')

In [0]:
df_silver.select("category_code").distinct().show()

+-------------+
|category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|        BOOKS|
|      GROCERY|
|         GRCY|
|          TOY|
|         TOYS|
|          SPT|
+-------------+



In [0]:
fix = {"GRCY":"GROCERY", "TOYS":"TOY"}
df_silver = df_silver.replace(fix, subset="category_code")
df_silver.select("category_code").distinct().show()

+-------------+
|category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|        BOOKS|
|      GROCERY|
|          TOY|
|          SPT|
+-------------+



In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.slv_brands")

## **CATEGORIES**

In [0]:
catalog_name = "ecommerce"
df_bronze = spark.table(f"{catalog_name}.bronze.brz_categories")
df_bronze.show(truncate=False)


+-------------+----------------------+----------------------------------------------------+--------------------------+
|category_code|category_name         |_source_file                                        |ingested_at               |
+-------------+----------------------+----------------------------------------------------+--------------------------+
|ce           |Electronics           |dbfs:/Volumes/ecommerce/source_data/raw/category.csv|2026-11-16 13:07:46.733342|
|app          |Apparel               |dbfs:/Volumes/ecommerce/source_data/raw/category.csv|2026-11-16 13:07:46.733342|
|hnk          |Home & Kitchen        |dbfs:/Volumes/ecommerce/source_data/raw/category.csv|2026-11-16 13:07:46.733342|
|bpc          |Beauty & Personal Care|dbfs:/Volumes/ecommerce/source_data/raw/category.csv|2026-11-16 13:07:46.733342|
|bks          |Books                 |dbfs:/Volumes/ecommerce/source_data/raw/category.csv|2026-11-16 13:07:46.733342|
|grcy         |Grocery               |dbfs:/Volu

In [0]:
from pyspark.sql.types import StructType,StructField, StringType, IntegerType,FloatType, DateType, TimestampType
from pyspark.sql import functions as F

In [0]:
dup = df_bronze.groupBy('category_code').count().filter('count > 1')
dup.show()

+-------------+-----+
|category_code|count|
+-------------+-----+
|          app|    2|
|         grcy|    2|
+-------------+-----+



In [0]:
df_bronze = df_bronze.dropDuplicates(['category_code'])

In [0]:
dup = df_bronze.groupBy('category_code').count().filter('count > 1')
dup.show()

+-------------+-----+
|category_code|count|
+-------------+-----+
+-------------+-----+



In [0]:
df_silver = df_bronze.withColumn("category_code", F.upper(F.col("category_code")))
display(df_silver)

category_code,category_name,_source_file,ingested_at
CE,Electronics,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:07:46.733Z
APP,Apparel,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:07:46.733Z
HNK,Home & Kitchen,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:07:46.733Z
BPC,Beauty & Personal Care,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:07:46.733Z
BKS,Books,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:07:46.733Z
GRCY,Grocery,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:07:46.733Z
TOY,Toys & Games,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:07:46.733Z
SPT,Sports & Outdoors,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:07:46.733Z


In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.slv_categories")

## **CUSTOMERS**

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_customers")
df_bronze.show()

+----------------+---------------+------------+--------------------+-----+--------------------+--------------------+
|     customer_id|          phone|country_code|             country|state|        _source_file|         ingested_at|
+----------------+---------------+------------+--------------------+-----+--------------------+--------------------+
|CUST000000000001| 917280033536.0|          IN|               India|   MH|dbfs:/Volumes/eco...|2026-11-16 13:50:...|
|CUST000000000002| 619489725433.0|          AU|           Australia|  VIC|dbfs:/Volumes/eco...|2026-11-16 13:50:...|
|CUST000000000003| 919390066524.0|          IN|               India|   TN|dbfs:/Volumes/eco...|2026-11-16 13:50:...|
|CUST000000000004| 917073741793.0|          IN|               India|   TN|dbfs:/Volumes/eco...|2026-11-16 13:50:...|
|CUST000000000005| 618478772532.0|          AU|           Australia|   WA|dbfs:/Volumes/eco...|2026-11-16 13:50:...|
|CUST000000000006| 916441718520.0|          IN|               In

In [0]:
%sql
select * from ecommerce.bronze.brz_customers where customer_id is null

customer_id,phone,country_code,country,state,_source_file,ingested_at
null,918187043562.0,IN,India,DL,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
null,917517243052.0,IN,India,DL,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
null,null,IN,India,GJ,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
null,447220214605.0,GB,United Kingdom,WLS,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
null,916996290632.0,IN,India,UP,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
null,919485111439.0,IN,India,DL,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
null,919041533418.0,IN,India,WB,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
null,448120086491.0,GB,United Kingdom,NIR,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
null,447765784336.0,GB,United Kingdom,ENG,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
null,916767187689.0,IN,India,KA,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z


In [0]:
df_bronze.count()

300000

In [0]:
df_silver = df_bronze.na.drop(how="all")
df_silver.count()

300000

In [0]:
df_silver = df_silver.na.drop(subset=['customer_id'])
df_silver.count()

299700

In [0]:
%sql
select * from ecommerce.bronze.brz_customers where country is null

customer_id,phone,country_code,country,state,_source_file,ingested_at


In [0]:
%sql
select * from ecommerce.bronze.brz_customers where phone is null

customer_id,phone,country_code,country,state,_source_file,ingested_at
CUST000000000007,null,IN,India,MH,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
CUST000000000010,null,IN,India,RJ,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
CUST000000000026,null,IN,India,WB,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
CUST000000000032,null,US,United States,NJ,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
CUST000000000070,null,IN,India,TS,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
CUST000000000081,null,AU,Australia,VIC,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
CUST000000000103,null,US,United States,CA,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
CUST000000000112,null,IN,India,MH,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
CUST000000000117,null,SG,Singapore,SG,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z
CUST000000000124,null,IN,India,MH,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:22.178Z


In [0]:
df_silver = df_bronze.na.fill("Not available", subset="phone")

In [0]:
df_silver.filter(df_silver['phone'].isNull()).count()

0

In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.slv_customers")

In [0]:
%sql
select * from ecommerce.silver.slv_customers where phone is null

customer_id,phone,country_code,country,state,_source_file,ingested_at


## **DATE**

In [0]:
catalog_name="ecommerce"
df_bronze = spark.table(f"{catalog_name}.bronze.brz_date")
df_bronze.show()

+----------+----+---------+-------+------------+--------------------+--------------------+
|      date|year| day_name|quarter|week_of_year|        _source_file|         ingested_at|
+----------+----+---------+-------+------------+--------------------+--------------------+
|01-08-2025|2025|   friday|      3|         -31|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|02-08-2025|2025| SATURDAY|      3|         -31|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|03-08-2025|2025|   SUNDAY|      3|         -31|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|04-08-2025|2025|   MONDAY|      3|         -32|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|05-08-2025|2025|  TUESDAY|      3|         -32|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|06-08-2025|2025|WEDNESDAY|      3|         -32|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|07-08-2025|2025| thursday|      3|         -32|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|08-08-2025|2025|   friday|      3|         -32|dbfs:/Volumes/eco...|2026-11-16 13:55:...|

In [0]:
from pyspark.sql.functions import to_date
# Convert the string column to a date type
df_silver = df_bronze.withColumn("date", to_date(df_bronze["date"], "dd-MM-yyyy"))

In [0]:
df_silver = df_silver.dropDuplicates(['date'])


In [0]:
from pyspark.sql import functions as F
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))
df_silver.show(5)

+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _source_file|         ingested_at|
+----------+----+--------+-------+------------+--------------------+--------------------+
|2025-08-01|2025|  Friday|      3|         -31|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-02|2025|Saturday|      3|         -31|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-03|2025|  Sunday|      3|         -31|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-04|2025|  Monday|      3|         -32|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-05|2025| Tuesday|      3|         -32|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
df_silver = df_silver.withColumn("week_of_year", F.abs(F.col('week_of_year')))
df_silver.show()

+----------+----+---------+-------+------------+--------------------+--------------------+
|      date|year| day_name|quarter|week_of_year|        _source_file|         ingested_at|
+----------+----+---------+-------+------------+--------------------+--------------------+
|2025-08-01|2025|   Friday|      3|          31|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-02|2025| Saturday|      3|          31|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-03|2025|   Sunday|      3|          31|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-04|2025|   Monday|      3|          32|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-05|2025|  Tuesday|      3|          32|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-06|2025|Wednesday|      3|          32|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-07|2025| Thursday|      3|          32|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-08|2025|   Friday|      3|          32|dbfs:/Volumes/eco...|2026-11-16 13:55:...|

In [0]:
from pyspark.sql.types import StructType,StructField, StringType, IntegerType,FloatType, DateType, TimestampType
from pyspark.sql import functions as F
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))

df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))

df_silver.show(3)

+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _source_file|         ingested_at|
+----------+----+--------+-------+------------+--------------------+--------------------+
|2025-08-01|2025|  Friday|Q3-2025| Week31-2025|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-02|2025|Saturday|Q3-2025| Week31-2025|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
|2025-08-03|2025|  Sunday|Q3-2025| Week31-2025|dbfs:/Volumes/eco...|2026-11-16 13:55:...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 3 rows


In [0]:
#rename columns
df_silver = df_silver.withColumnRenamed("week_of_year","week")


In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.silver.slv_calendar")

## **PRODUCTS**

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_products")
df_bronze.show()

+-------------+--------------+-------------+----------+------+--------+---------+------------+---------+--------+---------+------------+---------+----------------+--------------------+--------------------+
|   product_id|           sku|category_code|brand_code| color|    size| material|weight_grams|length_cm|width_cm|height_cm|rating_count|file_name|ingest_timestamp|        _source_file|         ingested_at|
+-------------+--------------+-------------+----------+------+--------+---------+------------+---------+--------+---------+------------+---------+----------------+--------------------+--------------------+
|2000000000015|STCR-HNK-00001|          hnk|      stcr| White|One-Size|    Coton|        305g|     22,2|    17.1|      6.3|           0|     NULL|            NULL|dbfs:/Volumes/eco...|2026-11-16 13:17:...|
|2000000000022|HMNS-HNK-00002|          hnk|      hmns|Silver|One-Size|    Steel|        682g|     18,2|    12.3|      3.7|           1|     NULL|            NULL|dbfs:/Volumes

In [0]:
df_bronze.count()

50000

In [0]:
df_silver = df_bronze.na.drop(how="all")
df_silver.count()

50000

In [0]:
df_silver = df_bronze.withColumn("weight_grams",F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType()))

In [0]:
df_silver = df_silver.withColumn("length_cm",F.regexp_replace(F.col("length_cm"), ",", ".").cast(FloatType()))

In [0]:
df_silver.select('category_code','brand_code').show()

+-------------+----------+
|category_code|brand_code|
+-------------+----------+
|          hnk|      stcr|
|          hnk|      hmns|
|           ce|      novw|
|          app|      urtl|
|         grcy|      ggrn|
|          bpc|      slke|
|           ce|      volt|
|          app|      cblt|
|          spt|      arft|
|          app|      mosa|
|          app|      cblt|
|          toy|      rkbr|
|          toy|      gmfg|
|         grcy|      grbk|
|          bpc|      dmlx|
|         grcy|      ggrn|
|           ce|      znth|
|          bpc|      btnq|
|           ce|      skyl|
|         grcy|      ggrn|
+-------------+----------+
only showing top 20 rows


In [0]:
from pyspark.sql import functions as F
df_silver = df_silver.withColumn("category_code", F.upper(F.col("category_code"))). withColumn("brand_code", F.upper(F.col("brand_code")))

In [0]:
df_silver.select("category_code").show()

+-------------+
|category_code|
+-------------+
|          HNK|
|          HNK|
|           CE|
|          APP|
|         GRCY|
|          BPC|
|           CE|
|          APP|
|          SPT|
|          APP|
|          APP|
|          TOY|
|          TOY|
|         GRCY|
|          BPC|
|         GRCY|
|           CE|
|          BPC|
|           CE|
|         GRCY|
+-------------+
only showing top 20 rows


In [0]:
display(df_silver)

product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp,_source_file,ingested_at
2000000000015,STCR-HNK-00001,HNK,STCR,White,One-Size,Coton,305,22.2,17.1,6.3,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:17:01.164Z
2000000000022,HMNS-HNK-00002,HNK,HMNS,Silver,One-Size,Steel,682,18.2,12.3,3.7,1,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:17:01.164Z
2000000000039,NOVW-CE-00003,CE,NOVW,Purple,One-Size,Wood,243,18.2,13.9,4.2,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:17:01.164Z
2000000000046,URTL-APP-00004,APP,URTL,Silver,S,Ruber,225,17.6,4.6,5.8,50,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:17:01.164Z
2000000000053,GGRN-GRC-00005,GRCY,GGRN,Silver,One-Size,Ruber,455,27.2,15.8,7.4,-4,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:17:01.164Z
2000000000060,SLKE-BPC-00006,BPC,SLKE,Purple,One-Size,Plastic,232,28.0,13.8,6.1,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:17:01.164Z
2000000000077,VOLT-CE-00007,CE,VOLT,Blue,One-Size,Plastic,507,27.2,12.1,6.4,5,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:17:01.164Z
2000000000084,CBLT-APP-00008,APP,CBLT,Blue,XS,Polyester,261,27.7,8.5,7.0,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:17:01.164Z
2000000000091,ARFT-SPT-00009,SPT,ARFT,Blue,XL,Plastic,59,12.5,19.0,7.9,11,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:17:01.164Z
2000000000107,MOSA-APP-0000A,APP,MOSA,White,L,Polyester,238,10.7,17.7,10.3,6,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:17:01.164Z


In [0]:
%sql 
select distinct material from ecommerce.bronze.brz_products

material
Coton
Steel
Wood
Ruber
Plastic
Polyester
Glass
Alumium
Paper
Leather


In [0]:
#fix spelling mistakes in materials
fix = {
    "Coton":"Cotton",
    "Ruber":"Rubber",
    "Alumium":"Aluminium"
}
df_silver = df_silver.replace(fix, subset='material')


In [0]:
df_silver.select('material').distinct().show()

+---------+
| material|
+---------+
|   Cotton|
|    Steel|
|     Wood|
|   Rubber|
|  Plastic|
|Polyester|
|    Glass|
|Aluminium|
|    Paper|
|  Leather|
+---------+



In [0]:
# Convert negative rating_count to positive
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0))  # if null, replace with 0
)

In [0]:
# Check final cleaned data

df_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10, truncate=False)

+------------+---------+-------------+----------+---------+------------+
|weight_grams|length_cm|category_code|brand_code|material |rating_count|
+------------+---------+-------------+----------+---------+------------+
|305         |22.2     |HNK          |STCR      |Cotton   |0           |
|682         |18.2     |HNK          |HMNS      |Steel    |1           |
|243         |18.2     |CE           |NOVW      |Wood     |0           |
|225         |17.6     |APP          |URTL      |Rubber   |50          |
|455         |27.2     |GRCY         |GGRN      |Rubber   |4           |
|232         |28.0     |BPC          |SLKE      |Plastic  |0           |
|507         |27.2     |CE           |VOLT      |Plastic  |5           |
|261         |27.7     |APP          |CBLT      |Polyester|0           |
|59          |12.5     |SPT          |ARFT      |Plastic  |11          |
|238         |10.7     |APP          |MOSA      |Polyester|6           |
+------------+---------+-------------+----------+--

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_dim_products)
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.silver.slv_products")